# Cross-Source PropAMM Quote Ladder Analysis

Polaris normalizes on-chain proprietary AMM observations into one
[PropAMM Quote Ladders](https://docs.polaris.supply/schemas/propamm-quote-ladders) schema. This
notebook checks all six documented sources, selects a directed token pair shared by the most sources,
and compares executable quote curves and price impact without losing uint256 precision.


## Setup

Quote ladders are sparse block-level events. Each source is queried over the first five minutes of
its no-key preview day, with a 5,000-row cap.


In [ ]:
from itertools import islice

from polaris_data import PolarisClient
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)


def as_utc(value):
    timestamp = pd.Timestamp(value)
    return timestamp.tz_localize("UTC") if timestamp.tzinfo is None else timestamp.tz_convert("UTC")


def accessible_bounds(market_info):
    """Return the no-key catalog interval for an open or preview market."""
    start = as_utc(market_info["start"])
    end = as_utc(market_info["end"])
    access = market_info.get("access") or {}
    cutoff = access.get("public_cutoff_date")
    if access.get("status") == "preview" and cutoff:
        public_day = as_utc(cutoff)
        start = max(start, public_day)
        end = min(end, public_day + pd.Timedelta(days=1))
    if start >= end:
        raise ValueError("Catalog metadata does not expose a no-key interval for this market")
    return start, end


def bounded_rows(iterator, limit):
    """Materialize at most limit rows and close a partially consumed SDK generator."""
    rows = list(islice(iterator, limit + 1))
    truncated = len(rows) > limit
    close = getattr(iterator, "close", None)
    if close is not None:
        close()
    return rows[:limit], truncated


def event_timestamp(row):
    """Support both the legacy and v2 Polaris event envelopes."""
    value = row.get("collector_timestamp", row.get("timestamp"))
    return pd.to_datetime(value, unit="ms", utc=True)

from decimal import Decimal


In [ ]:
sources = ["fermiswap", "bopamm", "kipseli", "metric", "tempest", "taurusfi"]
market = "ethereum"
window_length = pd.Timedelta(minutes=5)
max_rows_per_source = 5_000


## Discover and fetch each source

Empty sources remain in the coverage table so absence in a short public window is not confused with
an unsupported schema.


In [ ]:
rows_by_source = {}
coverage_records = []

with PolarisClient() as client:
    for source in sources:
        catalog = client.catalog(source=source, market=market)
        if not catalog.get("markets"):
            rows_by_source[source] = []
            coverage_records.append({"source": source, "rows": 0, "hit_cap": False, "window": "not in catalog"})
            continue

        market_info = catalog["markets"][0]
        start, accessible_end = accessible_bounds(market_info)
        end = min(start + window_length, accessible_end)
        rows, truncated = bounded_rows(
            client.propamm_quote_ladders(
                source=source, market=market, from_=start, to=end, allow_gaps=True,
            ),
            max_rows_per_source,
        )
        rows_by_source[source] = rows
        coverage_records.append({
            "source": source,
            "rows": len(rows),
            "hit_cap": truncated,
            "window": f"{start} -> {end}",
        })

coverage_df = pd.DataFrame(coverage_records)
coverage_df


## Inspect event identity and select a comparable pair

Comparison is restricted to the same token direction. A pair seen across the most sources wins;
ties prefer the pair with the most quote observations. If only one source is populated, its richest
pair still produces a complete single-source analysis.


In [ ]:
ladder_records = []
for source, rows in rows_by_source.items():
    for row in rows:
        values = (row.get("data") or {}).get("values") or {}
        ladder_records.append({
            "source": source,
            "timestamp": event_timestamp(row),
            "token_in": (values.get("token_in") or "").lower(),
            "token_out": (values.get("token_out") or "").lower(),
            "token_in_decimals": values.get("token_in_decimals"),
            "token_out_decimals": values.get("token_out_decimals"),
            "quotes": values.get("quotes") or [],
            "event_id": values.get("event_id"),
            "block_number": values.get("block_number"),
            "transaction_hash": values.get("transaction_hash"),
            "router": values.get("router"),
            "oracle": values.get("oracle"),
            "pool": values.get("pool"),
        })

ladders_df = pd.DataFrame(ladder_records)
if ladders_df.empty:
    raise ValueError("No PropAMM quote ladders were found in the public sample windows")

pair_coverage = (
    ladders_df.groupby(["token_in", "token_out"])
    .agg(sources=("source", "nunique"), ladders=("source", "size"), quote_points=("quotes", lambda values: sum(map(len, values))))
    .sort_values(["sources", "quote_points", "ladders"], ascending=False)
)
selected_pair = pair_coverage.index[0]
print(f"Selected directed pair: {selected_pair[0]} -> {selected_pair[1]}")
pair_coverage.head(10)


## Flatten the most recent comparable ladders

Amounts stay as `Decimal` while token decimals are applied. Average execution rate is compared with
each source's smallest quote to produce within-source impact in basis points.


In [ ]:
selected_ladders = (
    ladders_df[
        ladders_df["token_in"].eq(selected_pair[0])
        & ladders_df["token_out"].eq(selected_pair[1])
    ]
    .sort_values("timestamp")
    .drop_duplicates("source", keep="last")
)

quote_records = []
for _, ladder in selected_ladders.iterrows():
    input_scale = Decimal(10) ** int(ladder["token_in_decimals"])
    output_scale = Decimal(10) ** int(ladder["token_out_decimals"])
    for quote in ladder["quotes"]:
        amount_in = Decimal(quote["amount_in"]) / input_scale
        amount_out = Decimal(quote["amount_out"]) / output_scale
        if amount_in <= 0:
            continue
        quote_records.append({
            "source": ladder["source"],
            "amount_in": amount_in,
            "amount_out": amount_out,
            "average_rate": amount_out / amount_in,
        })

quotes_df = pd.DataFrame(quote_records)
baseline_rates = quotes_df.loc[
    quotes_df.groupby("source")["amount_in"].idxmin(), ["source", "average_rate"]
].rename(columns={"average_rate": "baseline_rate"})
quotes_df = quotes_df.merge(baseline_rates, on="source")
quotes_df["impact_bps"] = quotes_df.apply(
    lambda row: (row["average_rate"] / row["baseline_rate"] - Decimal(1)) * Decimal(10_000),
    axis=1,
)
for column in ["amount_in", "amount_out", "average_rate", "impact_bps"]:
    quotes_df[f"{column}_float"] = quotes_df[column].astype(float)

metadata_columns = [
    "source", "timestamp", "block_number", "transaction_hash", "router", "oracle", "pool", "event_id"
]
display(selected_ladders[metadata_columns])
quotes_df.groupby("source").agg(
    quote_points=("amount_in", "size"),
    min_input=("amount_in_float", "min"),
    max_input=("amount_in_float", "max"),
    worst_impact_bps=("impact_bps_float", "min"),
)


## Quote curves and price impact

Logarithmic input axes expose the full ladder. Because comparison is restricted to one directed pair,
rate and impact differences are meaningful across the populated PropAMMs.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for source, values in quotes_df.groupby("source"):
    values = values.sort_values("amount_in_float")
    axes[0].plot(values["amount_in_float"], values["amount_out_float"], marker=".", label=source)
    axes[1].plot(values["amount_in_float"], values["impact_bps_float"], marker=".", label=source)

axes[0].set_xscale("log")
axes[0].set_title("Normalized output quote curve")
axes[0].set_xlabel("Input token amount")
axes[0].set_ylabel("Output token amount")
axes[0].legend()

axes[1].set_xscale("log")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Average-rate impact vs smallest quote")
axes[1].set_xlabel("Input token amount")
axes[1].set_ylabel("Impact (bps)")
axes[1].legend()

plt.tight_layout()
plt.show()
